# D2O Flexible Model File Preparation
Converts TIP4P-style D2O (OW/DW1/DW2/MW) to 3-site flexible model (OW/DW1/DW2).
- Strips MW virtual sites from .gro
- Corrects molecule count in topology

In [ ]:
import re
from pathlib import Path

In [ ]:
GRO_IN  = Path('original_d2o.gro')
TOP_IN  = Path('original_d2o.top')
GRO_OUT = Path('d2o_flexible.gro')
TOP_OUT = Path('d2o_flexible.top')

## 1. Parse and strip MW from .gro

In [ ]:
def strip_mw_from_gro(path_in: Path, path_out: Path) -> dict:
    """
    Remove MW virtual-site lines from a GROMACS .gro file.
    Returns summary statistics.
    """
    lines = path_in.read_text().splitlines()

    title   = lines[0]
    n_atoms = int(lines[1].strip())
    box     = lines[-1]          # always last line
    atom_lines = lines[2:-1]     # exclude header, count, box

    assert len(atom_lines) == n_atoms, (
        f'Expected {n_atoms} atom lines, found {len(atom_lines)}'
    )

    kept   = []
    n_mw   = 0
    new_idx = 1
    res_map = {}  # old resnum -> new resnum
    new_res = 0

    for line in atom_lines:
        # .gro fixed format: cols 0-4 resnum, 5-9 resname, 10-14 atomname,
        # 15-19 atomnum, then x y z (and optionally vx vy vz)
        atom_name = line[10:15].strip()
        if atom_name == 'MW':
            n_mw += 1
            continue

        old_res = int(line[0:5])
        if old_res not in res_map:
            new_res += 1
            res_map[old_res] = new_res

        # Rebuild line with renumbered residue and atom indices
        new_line = (
            f'{res_map[old_res]:5d}'
            f'{line[5:15]}'
            f'{new_idx:5d}'
            f'{line[20:]}'
        )
        kept.append(new_line)
        new_idx += 1

    n_kept = len(kept)
    n_mol  = new_res

    out_lines = [title, f'{n_kept:5d}'] + kept + [box]
    path_out.write_text('\n'.join(out_lines) + '\n')

    return {
        'original_atoms': n_atoms,
        'mw_removed': n_mw,
        'kept_atoms': n_kept,
        'molecules': n_mol,
        'box': box.strip()
    }


stats = strip_mw_from_gro(GRO_IN, GRO_OUT)
for k, v in stats.items():
    print(f'{k:>20s}: {v}')

## 2. Fix molecule count in topology

In [ ]:
def fix_topology(path_in: Path, path_out: Path, n_mol: int) -> None:
    """
    Update the [molecules] section to use the correct molecule count.
    """
    text = path_in.read_text()

    # Replace molecule count line: 'D2O_flex  <any_number>' -> correct count
    updated = re.sub(
        r'(D2O_flex\s+)\d+',
        lambda m: m.group(1) + str(n_mol),
        text
    )

    if updated == text:
        raise RuntimeError('No D2O_flex molecule line found in topology.')

    path_out.write_text(updated)
    print(f'Molecule count set to {n_mol} in {path_out}')


fix_topology(TOP_IN, TOP_OUT, stats['molecules'])

## 3. Validate output .gro

In [ ]:
import numpy as np

def validate_gro(path: Path) -> None:
    """
    Check atom count, no MW present, and O-D bond lengths are reasonable.
    """
    lines = path.read_text().splitlines()
    n_atoms = int(lines[1].strip())
    atom_lines = lines[2:2 + n_atoms]

    atom_names = [l[10:15].strip() for l in atom_lines]
    assert 'MW' not in atom_names, 'MW atoms still present'

    coords = {}
    for line in atom_lines:
        res  = int(line[0:5])
        name = line[10:15].strip()
        xyz  = np.array([float(line[20:28]), float(line[28:36]), float(line[36:44])])
        coords.setdefault(res, {})[name] = xyz

    bond_lengths = []
    for mol in coords.values():
        if 'OW' in mol and 'DW1' in mol:
            bond_lengths.append(np.linalg.norm(mol['OW'] - mol['DW1']))
        if 'OW' in mol and 'DW2' in mol:
            bond_lengths.append(np.linalg.norm(mol['OW'] - mol['DW2']))

    bl = np.array(bond_lengths)
    print(f'Atoms in file:       {n_atoms}')
    print(f'Molecules validated: {len(coords)}')
    print(f'O-D bonds checked:   {len(bl)}')
    print(f'Bond length mean:    {bl.mean():.4f} nm')
    print(f'Bond length std:     {bl.std():.4f} nm')
    print(f'Bond length range:   [{bl.min():.4f}, {bl.max():.4f}] nm')
    print(f'Target O-D length:   0.1000 nm  (flexible topology)')

    # Warn if any bond deviates more than 20% from target
    outliers = np.sum(np.abs(bl - 0.10) > 0.02)
    if outliers:
        print(f'WARNING: {outliers} bonds deviate >20% from 0.10 nm')
    else:
        print('All O-D bonds within acceptable range.')


validate_gro(GRO_OUT)

## 4. Validate topology molecule count

In [ ]:
def validate_top(path: Path, expected_mol: int) -> None:
    text = path.read_text()
    m = re.search(r'D2O_flex\s+(\d+)', text)
    assert m, 'D2O_flex not found in topology'
    found = int(m.group(1))
    assert found == expected_mol, f'Expected {expected_mol} molecules, got {found}'
    print(f'Topology molecule count verified: {found}')
    print(f'Expected atoms (3 per mol):       {found * 3}')


validate_top(TOP_OUT, stats['molecules'])

## 5. Summary

In [ ]:
print('Output files ready for GROMACS energy minimization:')
print(f'  Coordinates: {GRO_OUT}')
print(f'  Topology:    {TOP_OUT}')
print()
print('Next step:')
print('  gmx grompp -f em.mdp -c d2o_flexible.gro -p d2o_flexible.top -o em.tpr')
print('  gmx mdrun -v -deffnm em')